# Term simplification via rewriting

In [ ]:
// Nested tuples, the representation of abstract syntax trees; loaded once per session.
#load "../Support/Nested.fsx"

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "AutomataTypes.fsx"
#load "RegExpParser.fsx"

In [ ]:
open AutomataTypes
open Nested
open RegExpParser

Regular expressions are nested tuples of the type `RegExp` (see `AutomataTypes.fsx`).  A rewrite rule is a pair of
regular expressions, its *left-hand side* and its *right-hand side*.  In a rule, every uppercase letter is a
*pattern variable*, which matches an arbitrary regular expression; the function `isVariable` recognizes them.  In a
term, an uppercase letter is just a letter.

In [ ]:
type Substitution = Map<string, RegExp>
type Rule = RegExp * RegExp

/// isVariable p checks whether p is a pattern variable, i.e. an uppercase letter
let isVariable (pattern: RegExp) =
    match pattern with
    | Leaf (Str s) -> s.Length = 1 && 'A' <= s[0] && s[0] <= 'Z'
    | _ -> false

A substitution is an immutable map from pattern-variable names to regular expressions.  Leaves match if they are equal, and tuples match if they have the same length and all their components match. Matching returns `Some substitution` when it succeeds and `None` otherwise. Repeated occurrences of a pattern variable must match exactly the same expression. Each match starts with a fresh map, avoiding the original mutable default argument.

In [ ]:
let rec matchPattern (pattern: RegExp) (term: RegExp) (substitution: Substitution) : Substitution option =
    match pattern, term with
    | Leaf (Str name), _ when isVariable pattern ->
        match Map.tryFind name substitution with
        | Some previous when previous = term -> Some substitution
        | Some _ -> None
        | None -> Some (Map.add name term substitution)
    | Leaf p, Leaf t when p = t -> Some substitution
    | Node patterns, Node terms when List.length patterns = List.length terms ->
        List.fold2 (fun result p t -> result |> Option.bind (matchPattern p t)) (Some substitution) patterns terms
    | _ -> None

`apply` instantiates a rule's right-hand side with the bindings from its left-hand side. `rewrite` tries one rule at the root of a term.

In [ ]:
let rec apply (pattern: RegExp) (substitution: Substitution) : RegExp =
    match pattern with
    | Leaf (Str name) when isVariable pattern ->
        Map.tryFind name substitution
        |> Option.defaultWith (fun () -> invalidArg "substitution" ("Unbound pattern variable: " + name))
    | Leaf _ -> pattern
    | Node components -> Node (List.map (fun p -> apply p substitution) components)

let rewrite term ((left, right): Rule) : RegExp option =
    matchPattern left term Map.empty |> Option.map (apply right)

These are the original 26 identities for alternatives, concatenation and Kleene star. Their order is explicit and deterministic. `R`, `S` and `T` are pattern variables; juxtaposition denotes concatenation.

In [ ]:
let getRules () : Rule list =
    [ "R+∅", "R"
      "∅+R", "R"
      "R+R", "R"
      "𝜀+R*", "R*"
      "R*+𝜀", "R*"
      "𝜀+RR*", "R*"
      "𝜀+R*R", "R*"
      "RR*+𝜀", "R*"
      "R*R+𝜀", "R*"
      "S+ST", "S(𝜀+T)"
      "S+TS", "(𝜀+T)S"
      "∅R", "∅"
      "R∅", "∅"
      "𝜀R", "R"
      "R𝜀", "R"
      "(𝜀+R)R*", "R*"
      "(R+𝜀)R*", "R*"
      "R*(R+𝜀)", "R*"
      "R*(𝜀+R)", "R*"
      "∅*", "𝜀"
      "𝜀*", "𝜀"
      "(𝜀+R)*", "R*"
      "(R+𝜀)*", "R*"
      "R+(S+T)", "(R+S)+T"
      "R(ST)", "(RS)T"
      "(RS*)(𝜀+S)", "RS*" ]
    |> List.map (fun (left, right) -> parse left, parse right)

`simplifyOnce` first tries a rule at the root, then recursively simplifies its children when no rule applies. `simplify` repeats the operation until the term no longer changes. Rules are parsed once per normalization. This is a simplifier, not a complete canonical form for regular languages.

In [ ]:
let rec simplifyOnce rules term =
    let rewritten = rules |> List.tryPick (fun rule -> rewrite term rule |> Option.filter ((<>) term))
    match rewritten with
    | Some result -> result
    | None ->
        match term with
        | Node components -> Node (List.map (simplifyOnce rules) components)
        | _ -> term

let simplify term =
    let rules = getRules ()
    let rec normalize expression =
        let next = simplifyOnce rules expression
        if next = expression then expression else normalize next
    normalize term

In [ ]:
let term = parse "(𝜀+a)a*"
checkNotebook (simplify term = parse "a*")
checkNotebook (simplify (parse "R+R") = literal 'R')
let repeated = parse "R+R"
checkNotebook (matchPattern repeated (parse "a+b") Map.empty = None)
simplify (parse "∅*+a𝜀")